# 用 Ollama 做图书网站摘要（Book Summaries）

## 练习目标（理念）

把第 1 周 Day 1 的「抓网页 + LLM 摘要」搬到**本地 Ollama**：用 `requests` + BeautifulSoup 抓取图书站点正文，再经 OpenAI 兼容接口调用 `llama3.2`，输出若干书的标题、作者与简介。

## 和本课 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Ollama OpenAI 兼容 | `base_url=http://localhost:11434/v1` |
| Chat Completions | `client.chat.completions.create(...)` |
| 网页清洗 | BeautifulSoup 去掉 script/style 等噪声 |
| system / user prompt | 书友会组长人设 + 摘要任务说明 |

## 怎么跑

1. 本机启动 Ollama，并 `ollama pull llama3.2`
2. 从上到下运行；最后一格会对 `https://thestorygraph.com` 做摘要（需能访问该站）


In [ ]:
# ========== 导入：抓网页 + 调 Ollama + 笔记本展示 ==========

# BeautifulSoup：解析 HTML，抽取标题与正文
from bs4 import BeautifulSoup
# requests：用 HTTP GET 下载网页
import requests
# OpenAI 客户端：这里指向本地 Ollama 的 OpenAI 兼容端点
from openai import OpenAI
# Markdown + display：把模型回复渲染成好看的 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 常量 + 创建指向 Ollama 的客户端 ==========

# 本地模型名：必须与 ollama list 里已有的名字一致
MODEL = "llama3.2"
# Ollama 的 OpenAI 兼容 Base URL（注意是 /v1，不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常只是占位；SDK 要求传一个非空字符串
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


In [ ]:
# ========== HTTP 请求头：伪装成常见浏览器，降低被站点拒绝的概率 ==========

# 许多站点会检查 User-Agent；这里沿用课程里的 Chrome UA 字符串（勿随意改，除非你知道后果）
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


In [ ]:
# ========== 抓取并清洗网页正文 ==========

def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    # GET 下载页面；带上 headers，减少裸请求被拒
    response = requests.get(url, headers=headers)
    # 用 html.parser 把字节/HTML 解析成可查询的 DOM 树
    soup = BeautifulSoup(response.content, "html.parser")
    # 取 <title>；没有则用占位英文（字符串保持原样）
    title = soup.title.string if soup.title else "No title found"
    # 有 <body> 才继续抽正文
    if soup.body:
        # 删掉脚本/样式/图片/输入框等对摘要无用的节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 抽出纯文本：块之间用换行分隔，并 strip 空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 没有 body 就当空文本
        text = ""
    # 标题 + 正文，并截断到 10_000 字符，避免 prompt 过长（docstring 仍写 2,000，逻辑保持原切片）
    return (title + "\n\n" + text)[:10_000]



In [ ]:
# ========== Prompt：书友会组长人设 + 摘要任务（英文指令保留） ==========

# system：定角色——本地读书会组长，负责选书/组织等
system_prompt = """
You are a leader in your local book club handling the usual
tasks of book club group leader i.e scheduling, meet-ups, researching
and coming up with books that will be read next etc
"""

# user：说明要从图书网站内容里抽出书名、作者与简介
user_prompt = """
Here are the contents of a books website.
Provide a short summary of a few books on the website.
Capture the book title, author and a short synopsis of each book.
"""


In [ ]:
# ========== summarize：抓页 → 拼 messages → Chat Completions ==========

def summarize(url):
    """抓取 url 正文，调用本地 llama3.2 生成图书摘要，返回助手回复字符串。"""
    # 先拿到清洗后的网站文本
    content = fetch_website_contents(url)
    # 调用 Chat Completions（非流式）：等整段生成完再返回
    response = client.chat.completions.create(
        model = MODEL,
        messages = [
            # system：角色设定
            {"role": "system", "content": system_prompt},
            # user：任务说明 + 网页正文（Website content: 前缀保持英文）
            {"role": "user", "content": user_prompt + f"\n\nWebsite content: {content}"},
        ]
    )
    # 取第一条 choice 的 message.content 作为摘要
    return response.choices[0].message.content


In [ ]:
# ========== 展示封装：summarize 后用 Markdown 渲染 ==========

def display_summary(url):
    """对 url 做摘要，并在笔记本输出区显示。"""
    # 调用上一格的 summarize
    summary = summarize(url)
    # 把返回字符串当 Markdown 展示
    display(Markdown(summary))


In [ ]:
# ========== 端到端：对 StoryGraph 图书站做摘要 ==========

# URL 保持原样；需本机 Ollama 已运行且能访问该网站
display_summary("https://thestorygraph.com")
